# Data Acquisition and Labelling

This notebook turns the raw N-BaIoT CSV files for the three chosen devices into one clean, labelled dataframe. It attaches the correct label to every row (read straight from the file name), checks the data for quality problems, ensures the labels are correct, shrinks the data to a manageable size, and saves the result for use in the next notebook.

In [1]:
# Standard library
import os
from pathlib import Path

# Third-party
import pandas

# Move into the repository root so config.py and every relative path inside it resolve correctly
if not Path("config.py").exists():  
    os.chdir("..")

# Local
import config

## Load and label the raw data

Each device's raw traffic sits in its own folder as several CSV files: one for normal (benign) traffic, and several more for each attack type. The label is not stored inside the file. Instead, it comes from the file name (for example, `mirai.ack.csv`). `config.load_device` reads every CSV for a device, identifies its label from the file name, and returns one combined and labelled dataframe.

This is done for every device in `config.CHOSEN_DEVICES` and the results are stacked together.

In [2]:
# Load and label every CSV file for each chosen device
device_frames = [config.load_device(Path(config.RAW_DIRECTORY) / device_name, device_name) for device_name in config.CHOSEN_DEVICES]
df = pandas.concat(device_frames, ignore_index=True)

## Check the data for quality problems

Before trusting this data, three common problems are checked:
- Missing values
- Duplicate rows
- Constant columns

How the three common problems are handled:
- Missing values in feature columns are filled with the median value (rows with a missing value in the label columns are dropped as the lack of a label makes it impossible to use).
- Duplicate rows and constant feature columns are dropped directly.

In [3]:
# Report and handle missing values
missing_counts = df.isnull().sum()
missing_columns = missing_counts[missing_counts > 0]
print(f"Missing values per column: {missing_columns if not missing_columns.empty else 'None'}")
missing_meta_rows = (df[config.META_COLUMNS].isnull().any(axis=1))
df = df[~missing_meta_rows]
missing_feature_columns = [column for column in config.feature_columns(df) if df[column].isnull().any()]
for column in missing_feature_columns:
    df[column] = df[column].fillna(df[column].median())

# Report and drop exact duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")
df = df.drop_duplicates()

# Report and drop constant-value feature columns
constant_columns = [column for column in config.feature_columns(df) if df[column].nunique() == 1]
print(f"Constant feature columns: {constant_columns if constant_columns else 'None'}")
df = df.drop(columns=constant_columns)

Missing values per column: None
Duplicate rows: 61349
Constant feature columns: None


## Confirm the labels are correct

Before proceeding, the row counts for each device/label combination and each specific attack subtype are printed to identify labelling mistakes.

In [4]:
# Perform a sanity check on row counts by device/label and by attack subtype
print(df.groupby(["device", "label"]).size())
print()
print(df["attack_subtype"].value_counts())

device                            label 
Danmini_Doorbell                  benign     40395
                                  gafgyt    304044
                                  mirai     652100
Philips_B120N10_Baby_Monitor      benign    165141
                                  gafgyt    300081
                                  mirai     610714
Provision_PT_838_Security_Camera  benign     93995
                                  gafgyt    296710
                                  mirai     429337
dtype: int64

attack_subtype
mirai.udp         613307
mirai.scan        308402
mirai.syn         302552
none              299531
gafgyt.udp        299303
gafgyt.tcp        253542
mirai.ack         251315
mirai.udpplain    216575
gafgyt.combo      175400
gafgyt.junk        86485
gafgyt.scan        86105
Name: count, dtype: int64


## Shrink the data to a manageable size

The full dataset contains too many rows and will slow down later steps.

To shrink the data, every benign row is kept because it is the only data the models are allowed to learn from. Each attack subtype is capped at `config.ATTACK_CAP` rows per device since attack data is only used for testing at the end.

`config.RANDOM_SEED` is used during sampling to ensure that results are reproducible.

In [5]:
# Keep all benign rows and cap each attack subtype per device
sampled_frames = [config.subsample_device(device_group) for _, device_group in df.groupby("device")]
sampled_df = pandas.concat(sampled_frames, ignore_index=True)
print(f"Rows before sampling: {len(df)}")
print(f"Rows after sampling: {len(sampled_df)}")

Rows before sampling: 2892517
Rows after sampling: 419531


## Save processed data

The result is saved as a single Parquet file for the next notebook to read. The label and attack-subtype columns remain in the file but are never given to a model as an input.

In [6]:
# Write the cleaned, labelled, sampled dataframe to disk
processed_directory = Path(config.PROCESSED_DIRECTORY)
processed_directory.mkdir(parents=True, exist_ok=True)
sampled_df.to_parquet(processed_directory / "df.parquet", index=False)